In [ ]:
import pandas as pd

df = pd.read_excel(r"/kaggle/working/datasets-dgtx/DATA_DG.xlsx")   # hoặc path đúng của bạn

assert {"content", "summary", "grade"}.issubset(df.columns)
df = df.dropna().reset_index(drop=True)


from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "VietAI/vit5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(tokenizer.model_max_length)  # ~512 tokens

from datasets import Dataset

MAX_INPUT_LEN = 256
MAX_TARGET_LEN = 64

def preprocess(batch):
    prompts = [
        f"Tóm tắt văn bản cho học sinh lớp {g}: {c}"
        for c, g in zip(batch["content"], batch["grade"])
    ]

    model_inputs = tokenizer(
        prompts,
        truncation=True,
        padding="max_length",
        max_length=MAX_INPUT_LEN,
    )

    labels = tokenizer(
        batch["summary"],
        truncation=True,
        max_length=MAX_TARGET_LEN,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

tokenized_ds = dataset.map(
    preprocess,
    batched=True,   # ⭐ BẮT BUỘC
    remove_columns=dataset["train"].column_names
)

print(tokenized_ds["train"].column_names)

In [ ]:
# ============================================
# CELL 1: OPTUNA HYPERPARAMETER TUNING
# ============================================

import optuna
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
import torch

def objective(trial):
    lr = trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True)
    batch_size = 1
    epochs = trial.suggest_int("num_train_epochs", 3, 5)
    weight_decay = trial.suggest_float("weight_decay", 0.0, 0.1)

    torch.cuda.empty_cache()
    import gc
    gc.collect()

    args = TrainingArguments(
        output_dir=f"./optuna_trial_{trial.number}",
        eval_strategy="no",
        logging_strategy="steps",
        logging_steps=100,
        save_strategy="no",
        learning_rate=lr,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=16,
        num_train_epochs=epochs,
        weight_decay=weight_decay,
        label_smoothing_factor=0.1,
        max_grad_norm=1.0,
        fp16=True,
        gradient_checkpointing=True,
        dataloader_num_workers=0,
        dataloader_pin_memory=False,
        ddp_find_unused_parameters=False,
        report_to="none"
    )

    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding=True,
        pad_to_multiple_of=None
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized_ds["train"],
        eval_dataset=tokenized_ds["test"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=None,
    )

    trainer.train()
    train_loss = trainer.state.log_history[-1].get("train_loss", float('inf'))
    
    del model
    del trainer
    del data_collator
    del args
    torch.cuda.empty_cache()
    import gc
    gc.collect()
    
    return train_loss

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=10)

best_params = study.best_params
print("=" * 50)
print("BEST HYPERPARAMETERS:")
print("=" * 50)
for key, value in best_params.items():
    print(f"{key}: {value}")
print("=" * 50)

In [ ]:
import evaluate
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

def compute_metrics(eval_pred):
    preds, labels = eval_pred

    import numpy as np
    
    # preds từ Trainer là logits, cần lấy argmax để có token IDs
    if isinstance(preds, tuple):
        preds = preds[0]
    
    if isinstance(preds, np.ndarray):
        # Lấy token ID có xác suất cao nhất (argmax)
        if preds.ndim == 3:  # (batch_size, seq_len, vocab_size)
            preds = np.argmax(preds, axis=-1)
        elif preds.ndim > 3:
            preds = preds.reshape(preds.shape[0], -1)
            preds = np.argmax(preds, axis=-1)
    
    # Xử lý labels - loại bỏ padding tokens (-100)
    if isinstance(labels, np.ndarray):
        labels = labels.tolist()
    
    # Decode predictions
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    
    # Decode labels - loại bỏ -100 trước khi decode
    decoded_labels = []
    for label in labels:
        if isinstance(label, np.ndarray):
            label = label.tolist()
        # Loại bỏ -100 (ignore index)
        valid_label = [l for l in label if l != -100]
        decoded_labels.append(tokenizer.decode(valid_label, skip_special_tokens=True))

    # Đảm bảo decoded_preds và decoded_labels là list các string
    decoded_preds = [str(p) if p else "" for p in decoded_preds]
    decoded_labels = [str(l) if l else "" for l in decoded_labels]

    # -------- ROUGE --------
    rouge_result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )

    # -------- BERTScore --------
    bert_result = bertscore.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        lang="vi",
        model_type="bert-base-multilingual-cased"
    )

    return {
        "rouge1": rouge_result["rouge1"],
        "rouge2": rouge_result["rouge2"],
        "rougeL": rouge_result["rougeL"],
        "bertscore_precision": sum(bert_result["precision"]) / len(bert_result["precision"]),
        "bertscore_recall": sum(bert_result["recall"]) / len(bert_result["recall"]),
        "bertscore_f1": sum(bert_result["f1"]) / len(bert_result["f1"]),
    }
    
import pandas as pd

def save_metrics(trainer, filename="metrics.csv"):
    logs = trainer.state.log_history
    df_metrics = pd.DataFrame(logs)
    df_metrics.to_csv(filename, index=False)
    return df_metrics

In [ ]:
# ============================================
# CELL 2: FINAL TRAINING VỚI BEST PARAMETERS
# ============================================

from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq, EarlyStoppingCallback
import torch
import os

torch.cuda.empty_cache()

# Kiểm tra xem có checkpoint để resume không
resume_from_checkpoint = None
checkpoint_dir = "./vit5_final"
if os.path.exists(checkpoint_dir):
    checkpoints = [f for f in os.listdir(checkpoint_dir) if f.startswith("checkpoint-")]
    if checkpoints:
        # Lấy checkpoint mới nhất
        checkpoint_numbers = [int(c.split("-")[1]) for c in checkpoints]
        latest_checkpoint = f"checkpoint-{max(checkpoint_numbers)}"
        resume_from_checkpoint = os.path.join(checkpoint_dir, latest_checkpoint)
        print(f"Tìm thấy checkpoint: {resume_from_checkpoint}")
        print("Sẽ tiếp tục training từ checkpoint này...")
    else:
        print("Không tìm thấy checkpoint, bắt đầu training từ đầu")
else:
    print("Không tìm thấy thư mục checkpoint, bắt đầu training từ đầu")

best_learning_rate = 0.0003290826760268271
best_num_train_epochs = 5
best_weight_decay = 0.004878586459798251

final_args = TrainingArguments(
    output_dir="./vit5_final",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=best_learning_rate,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=best_num_train_epochs,
    weight_decay=best_weight_decay,
    label_smoothing_factor=0.1,
    max_grad_norm=1.0,
    fp16=True,
    gradient_checkpointing=True,
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    ddp_find_unused_parameters=False,
    load_best_model_at_end=True,
    metric_for_best_model="eval_rougeL",
    greater_is_better=True,
    save_total_limit=5,
    report_to="none"
)

# Load model từ checkpoint nếu có, nếu không thì từ pretrained
if resume_from_checkpoint:
    final_model = AutoModelForSeq2SeqLM.from_pretrained(resume_from_checkpoint)
else:
    final_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

final_data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=final_model,
    padding=True,
    pad_to_multiple_of=None
)

final_trainer = Trainer(
    model=final_model,
    args=final_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    tokenizer=tokenizer,
    data_collator=final_data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("=" * 50)
print("BẮT ĐẦU TRAINING VỚI BEST PARAMETERS")
if resume_from_checkpoint:
    print(f"Resume từ checkpoint: {resume_from_checkpoint}")
print("=" * 50)

if resume_from_checkpoint:
    final_trainer.train(resume_from_checkpoint=resume_from_checkpoint)
else:
    final_trainer.train()

final_trainer.save_model("./vit5_grade_summary")
tokenizer.save_pretrained("./vit5_grade_summary")

print("=" * 50)
print("TRAINING HOÀN TẤT!")
print("Model đã được lưu tại: ./vit5_grade_summary")
print("=" * 50)

metrics_df = save_metrics(final_trainer, "vit5_metrics.csv")
print("\nMetrics:")
print(metrics_df.head())
print("\nColumns:", metrics_df.columns.tolist())

import matplotlib.pyplot as plt

eval_df = metrics_df[metrics_df["eval_rouge1"].notna()].copy()
if "epoch" not in eval_df.columns or eval_df["epoch"].isna().all():
    if "step" in eval_df.columns:
        steps_per_epoch = len(tokenized_ds["train"]) // (final_args.per_device_train_batch_size * final_args.gradient_accumulation_steps)
        eval_df["epoch"] = eval_df["step"] / steps_per_epoch
    else:
        eval_df["epoch"] = range(len(eval_df))

if len(eval_df) > 0:
    plt.figure(figsize=(10,6))
    if "eval_rouge1" in eval_df.columns:
        plt.plot(eval_df["epoch"], eval_df["eval_rouge1"], label="ROUGE-1")
    if "eval_rougeL" in eval_df.columns:
        plt.plot(eval_df["epoch"], eval_df["eval_rougeL"], label="ROUGE-L")
    bertscore_col = "eval_bertscore_f1" if "eval_bertscore_f1" in eval_df.columns else "bertscore_f1"
    if bertscore_col in eval_df.columns:
        plt.plot(eval_df["epoch"], eval_df[bertscore_col], label="BERTScore-F1")
    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.title("Evaluation Metrics over Epochs")
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print("Warning: No evaluation data found for plotting")

In [ ]:
# ============================================
# CELL: DỌN DẸP CHECKPOINT - XÓA FILE KHÔNG CẦN THIẾT
# ============================================

import os
import shutil

def clean_checkpoint_for_inference(checkpoint_dir, keep_training_files=False):
    """
    Xóa các file không cần thiết cho inference từ checkpoint
    
    Args:
        checkpoint_dir: Đường dẫn đến checkpoint (ví dụ: "./vit5_final/checkpoint-100")
        keep_training_files: Nếu True, giữ lại các file training (optimizer, scheduler, etc.)
    """
    if not os.path.exists(checkpoint_dir):
        print(f"Checkpoint không tồn tại: {checkpoint_dir}")
        return
    
    files_to_remove = []
    
    if not keep_training_files:
        training_files = [
            "optimizer.pt",
            "scheduler.pt", 
            "scaler.pt",
            "rng_state.pth"
        ]
        files_to_remove.extend(training_files)
    
    optional_files = [
        "trainer_state.json",
        "training_args.bin"
    ]
    files_to_remove.extend(optional_files)
    
    removed_count = 0
    for filename in files_to_remove:
        filepath = os.path.join(checkpoint_dir, filename)
        if os.path.exists(filepath):
            os.remove(filepath)
            removed_count += 1
            print(f"Đã xóa: {filename}")
    
    print(f"\nĐã xóa {removed_count} file không cần thiết")
    print(f"Checkpoint đã được tối ưu cho inference")

# Sử dụng: Chọn checkpoint cần dọn dẹp
# clean_checkpoint_for_inference("./vit5_final/checkpoint-100", keep_training_files=False)

# Hoặc dọn dẹp tất cả checkpoints trong thư mục
def clean_all_checkpoints(base_dir="./vit5_final", keep_training_files=False):
    """Dọn dẹp tất cả checkpoints trong thư mục"""
    if not os.path.exists(base_dir):
        print(f"Thư mục không tồn tại: {base_dir}")
        return
    
    checkpoints = [f for f in os.listdir(base_dir) if f.startswith("checkpoint-")]
    
    if not checkpoints:
        print("Không tìm thấy checkpoint nào")
        return
    
    print(f"Tìm thấy {len(checkpoints)} checkpoint(s)")
    
    for checkpoint in checkpoints:
        checkpoint_path = os.path.join(base_dir, checkpoint)
        print(f"\nĐang dọn dẹp: {checkpoint}")
        clean_checkpoint_for_inference(checkpoint_path, keep_training_files)
    
    print("\n" + "="*50)
    print("HOÀN TẤT DỌN DẸP!")

# Chạy để dọn dẹp (bỏ comment để sử dụng)
# clean_all_checkpoints("./vit5_final", keep_training_files=False)

In [10]:
# ============================================
# CELL: TEST MODEL
# ============================================

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import os

# Đường dẫn đến model đã train
MODEL_PATH = "../../ai/Model Train/Model_DG/vit5_grade_summary"
ORIGINAL_MODEL = "VietAI/vit5-base"

# Kiểm tra xem model có tồn tại không
if not os.path.exists(MODEL_PATH):
    print(f"Model không tồn tại tại: {MODEL_PATH}")
    print("Vui lòng train model trước hoặc kiểm tra đường dẫn")
else:
    print(f"Đang load model từ: {MODEL_PATH}")
    
    # Load tokenizer - thử từ checkpoint trước, nếu lỗi thì dùng model gốc
    # (Tokenizer không thay đổi sau training, chỉ model weights thay đổi)
    try:
        print("Đang load tokenizer từ checkpoint...")
        test_tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=False)
        print("✓ Tokenizer loaded từ checkpoint (slow tokenizer)")
    except Exception as e1:
        try:
            print(f"⚠ Không thể load slow tokenizer: {str(e1)[:100]}")
            print("Đang thử load fast tokenizer...")
            test_tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=True)
            print("✓ Tokenizer loaded từ checkpoint (fast tokenizer)")
        except Exception as e2:
            print(f"⚠ Không thể load tokenizer từ checkpoint: {str(e2)[:100]}")
            print(f"⚠ File tokenizer.json có thể bị corrupt")
            print(f"Đang load tokenizer từ model gốc {ORIGINAL_MODEL}...")
            print("(Lưu ý: Tokenizer không thay đổi sau training, nên dùng model gốc vẫn đúng)")
            test_tokenizer = AutoTokenizer.from_pretrained(ORIGINAL_MODEL)
            print("✓ Tokenizer loaded từ model gốc")
    
    # Load model weights từ checkpoint (đây là phần quan trọng - model đã được fine-tune)
    print("\nĐang load model weights từ checkpoint...")
    test_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)
    print("✓ Model weights loaded từ checkpoint")
    
    # Chuyển model sang GPU nếu có
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    test_model.to(device)
    test_model.eval()
    
    print(f"Model đã được load vào: {device}")
    print("=" * 50)

def summarize(text, grade, max_new_tokens=None, use_sampling=True, temperature=0.7):
    """
    Tóm tắt văn bản cho học sinh theo lớp
    
    Args:
        text: Văn bản cần tóm tắt
        grade: Lớp học (1-5)
        max_new_tokens: Số token tối đa
        use_sampling: Nếu True, dùng sampling để tạo output đa dạng
        temperature: Độ ngẫu nhiên khi use_sampling=True (0.1-1.0)
    
    Returns:
        Tóm tắt văn bản (đã được cắt ở câu hoàn chỉnh)
    """
    import re
    
    if grade < 1:
        grade = 1
    if grade > 5:
        grade = 5
    
    if max_new_tokens is None:
        # Tăng max_new_tokens đáng kể để đảm bảo đủ dài
        max_new_tokens = {
            1: 200,
            2: 350,
            3: 500,
            4: 600,
            5: 800  # Tăng lên 1000 cho lớp 5
        }.get(grade, grade * 180)
    
    # Tính min_new_tokens để đảm bảo tóm tắt đủ dài (khoảng 30% max)
    min_new_tokens = int(max_new_tokens * 0.3)
    
    prompt = f"Tóm tắt văn bản cho học sinh lớp {grade}: {text}"
    
    inputs = test_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True
    ).to(device)
    
    with torch.no_grad():
        if use_sampling:
            # Dùng sampling để tạo output đa dạng
            output = test_model.generate(
                **inputs,
                min_new_tokens=min_new_tokens,  # Đảm bảo tóm tắt đủ dài
                max_new_tokens=max_new_tokens,
                num_beams=1,
                length_penalty=1.2,  # Tăng để khuyến khích tóm tắt dài hơn
                repetition_penalty=1.4,  # Tăng để giảm lặp lại
                no_repeat_ngram_size=4,  # Tăng để tránh lặp cụm từ
                do_sample=True,
                temperature=temperature,  # Temperature cao hơn = đa dạng hơn
                top_p=0.9,  # Giảm một chút để ổn định hơn
                top_k=50,
                eos_token_id=test_tokenizer.eos_token_id,
                pad_token_id=test_tokenizer.pad_token_id
            )
        else:
            # Dùng beam search (ổn định nhưng giống nhau)
            output = test_model.generate(
                **inputs,
                min_new_tokens=min_new_tokens,
                max_new_tokens=max_new_tokens,
                num_beams=4,
                length_penalty=1.2,  # Tăng để khuyến khích tóm tắt dài hơn
                repetition_penalty=1.4,  # Tăng để giảm lặp lại
                no_repeat_ngram_size=4,  # Tăng để tránh lặp cụm từ
                early_stopping=False,
                do_sample=False,
                eos_token_id=test_tokenizer.eos_token_id,
                pad_token_id=test_tokenizer.pad_token_id
            )
    
    summary = test_tokenizer.decode(output[0], skip_special_tokens=True)
    
    # Post-processing: Cắt ở câu hoàn chỉnh và loại bỏ phần lặp lại
    # Tìm các dấu kết thúc câu
    sentences = re.split(r'([.!?]\s+)', summary)
    
    # Ghép lại các câu hoàn chỉnh
    complete_sentences = []
    for i in range(0, len(sentences) - 1, 2):
        if i + 1 < len(sentences):
            sentence = (sentences[i] + sentences[i + 1]).strip()
            if sentence:
                complete_sentences.append(sentence)
    
    # Nếu có câu cuối không có dấu kết thúc, thêm vào
    if len(sentences) % 2 == 1 and sentences[-1].strip():
        complete_sentences.append(sentences[-1].strip())
    
    # Loại bỏ các câu lặp lại (so sánh nội dung tương tự)
    filtered_sentences = []
    seen_content = set()
    for sentence in complete_sentences:
        # Chuẩn hóa để so sánh (lowercase, bỏ dấu cách thừa)
        normalized = re.sub(r'\s+', ' ', sentence.lower().strip())
        # Kiểm tra xem có quá giống với câu trước không (tránh lặp lại)
        is_duplicate = False
        for seen in seen_content:
            # Nếu giống nhau > 80% thì coi là duplicate
            if len(normalized) > 0 and len(seen) > 0:
                similarity = len(set(normalized.split()) & set(seen.split())) / max(len(normalized.split()), len(seen.split()))
                if similarity > 0.8:
                    is_duplicate = True
                    break
        
        if not is_duplicate:
            filtered_sentences.append(sentence)
            seen_content.add(normalized)
    
    # Ghép lại thành tóm tắt hoàn chỉnh
    if filtered_sentences:
        summary = ' '.join(filtered_sentences)
    else:
        # Nếu không có câu nào, dùng summary gốc
        summary = summary.strip()
    
    return summary


# ============================================
# TEST VỚI CÁC VÍ DỤ
# ============================================

print("\n" + "=" * 50)
print("TEST MODEL")
print("=" * 50)
test_text_3 = """Chu Văn An là một nhà giáo nổi tiếng đời Trần. Cụ đỗ cao nhưng không làm quan mà mở trường dạy học ở quê nhà nhằm truyền bá đạo lí và đào tạo nhân tài cho đất nước. Trường của cụ rất đông học trẻ, có nhiều người trở thành những nhân vật nổi tiếng.
Năm ấy, đến ngày mừng thọ cụ giáo Chu tròn sáu mươi tuổi, từ sáng sớm, các môn sinh đã tề tựu trước sân nhà cụ. Cụ Chu đội khăn ngay ngắn, mặc áo dài thâm ngồi trên sập. Mấy học trò cũ từ xa về dâng biếu thầy những cuốn sách quý do chính họ sưu tầm và chép lại. Cụ hỏi thăm công việc của từng người, bảo ban các học trò nhỏ, rồi đột nhiên nói:
– Thầy cảm ơn các anh. Bây giờ, thầy muốn mời tất cả các anh theo thầy tới thăm một người mà thầy mang ơn sâu nặng.
Các môn sinh đồng thanh dạ ran. Thế là thầy đi trước, trẻ theo sau. Các anh có tuổi đi ngay sau thầy, người ít tuổi hơn nhường bước, mấy chú tóc để trái đào đi sau cùng. Cụ dẫn học trò đi về cuối làng, đến một ngôi nhà tranh đơn sơ nhưng sáng sủa, ấm cúng. Ở hiên trước, một cụ già trên tám mươi tuổi, râu tóc bạc phơ đang ngồi sưởi nắng. Cụ giáo Chu bước vào sân, chắp tay cung kính vái và nói to:
– Lạy thầy! Hôm nay con đem tất cả môn sinh đến tạ ơn thầy.
Cụ già tóc bạc ngước lên, nghiêng đầu nghe. Cụ đã nặng tai. Thầy giáo Chu nói lại thật to câu nói vừa rồi một lần nữa. Thì ra đây là cụ đồ xưa kia đã dạy vỡ lòng cho cụ giáo Chu.
Tiếp sau cụ giáo Chu, các môn sinh lần lượt theo lứa tuổi vái tạ cụ đồ già. Ngày mừng thọ thầy Chu năm ấy, họ được thêm một bài học thấm thía về nghĩa thầy trò."""
test_grade_3 = 5

print(f"\n" + "-" * 50)
print(f"[TEST 3] Lớp {test_grade_3}")
print(f"Văn bản gốc: {test_text_3}")
print(f"\nTóm tắt:")
summary_3 = summarize(test_text_3, test_grade_3)
print(summary_3)

print("\n" + "=" * 50)
print("HOÀN TẤT TEST!")
print("=" * 50)

Đang load model từ: ../../ai/Model Train/Model_DG/vit5_grade_summary
Đang load tokenizer từ checkpoint...
✓ Tokenizer loaded từ checkpoint (slow tokenizer)

Đang load model weights từ checkpoint...
✓ Model weights loaded từ checkpoint
Model đã được load vào: cuda

TEST MODEL

--------------------------------------------------
[TEST 3] Lớp 5
Văn bản gốc: Chu Văn An là một nhà giáo nổi tiếng đời Trần. Cụ đỗ cao nhưng không làm quan mà mở trường dạy học ở quê nhà nhằm truyền bá đạo lí và đào tạo nhân tài cho đất nước. Trường của cụ rất đông học trẻ, có nhiều người trở thành những nhân vật nổi tiếng.
Năm ấy, đến ngày mừng thọ cụ giáo Chu tròn sáu mươi tuổi, từ sáng sớm, các môn sinh đã tề tựu trước sân nhà cụ. Cụ Chu đội khăn ngay ngắn, mặc áo dài thâm ngồi trên sập. Mấy học trò cũ từ xa về dâng biếu thầy những cuốn sách quý do chính họ sưu tầm và chép lại. Cụ hỏi thăm công việc của từng người, bảo ban các học trò nhỏ, rồi đột nhiên nói:
– Thầy cảm ơn các anh. Bây giờ, thầy muốn mời tất cả

In [11]:
print(summary_3)

Chu Văn An là nhà giáo nổi tiếng đời Trần, không làm quan mà mở trường dạy ở quê để truyền đạo lý và đào tạo nhân tài. Trường của cụ rất đông học sinh với nhiều người trở thành danh nhân. Năm ấy vào dịp mừng thọ sáu mươi tuổi, các môn Sinh tề tựu đưa sách quý đến biếu thầy, nhưng sau đó GS chu đáo nói: "Lớp cảm ơn Thầy!". Cụ hỏi thăm công việc và những cuốn vở đã được viết lại, mọi người nhận ra rằng ý nghĩacủa việc này mang theo những bài học thấm thía về lòng yêu thương. Cuối cùng, hôm nay thầy cho tất cả Môn sinh đến tạ ơn ông bằng cách mời các anh đi trước, nhường bước cho thầy. Các bạn nhỏ từ xa đến dâng quà tưởng nhớ thầy. Một ngày nọ, thầy nghe lời động viên của thầy khi đón tiếp chúng. Sau đó, họ gặp gỡ các môn sinh trong ngôi nhà tranh đơn sơ, ấm áp. Mỗi đứa trẻ một câu chuyện thú vị - Từ bốn mươi tám trăm triệu đồng, mỗi người đến tạ ơn. Lại thêm điều mới mẻ, hóa đá! Việc làm rõ ràng hơn nữa. Hôm nay, thầy sẽ trở


In [ ]:
# ============================================
# CELL: TEST BATCH PROCESSING
# ============================================

def summarize_batch(texts, grades, batch_size=4, max_new_tokens=100):
    """
    Tóm tắt nhiều văn bản cùng lúc (batch processing)
    
    Args:
        texts: List các văn bản cần tóm tắt
        grades: List các lớp tương ứng (cùng độ dài với texts)
        batch_size: Số lượng văn bản xử lý mỗi lần
        max_new_tokens: Số token tối đa cho mỗi tóm tắt
    
    Returns:
        List các tóm tắt
    """
    test_model.eval()
    results = []
    
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        batch_grades = grades[i:i+batch_size]
        
        prompts = [
            f"Tóm tắt văn bản cho học sinh lớp {g}: {t}"
            for t, g in zip(batch_texts, batch_grades)
        ]
        
        inputs = test_tokenizer(
            prompts,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)
        
        with torch.no_grad():
            outputs = test_model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                num_beams=4,
                length_penalty=1.3,
                repetition_penalty=1.2,
                no_repeat_ngram_size=3,
                early_stopping=True,
                do_sample=False
            )
        
        summaries = test_tokenizer.batch_decode(outputs, skip_special_tokens=True)
        results.extend(summaries)
    
    return results

# Test batch processing
print("=" * 50)
print("TEST BATCH PROCESSING")
print("=" * 50)

test_texts = [
    "Cây xanh giúp không khí trong lành và bảo vệ môi trường sống của con người.",
    "Nước là nguồn tài nguyên quý giá. Con người cần nước để uống, tắm rửa, nấu ăn.",
    "Việt Nam là một đất nước có bề dày lịch sử hàng nghìn năm."
]

test_grades = [3, 2, 5]

print(f"\nĐang xử lý {len(test_texts)} văn bản...")
batch_summaries = summarize_batch(test_texts, test_grades, batch_size=2)

for i, (text, grade, summary) in enumerate(zip(test_texts, test_grades, batch_summaries), 1):
    print(f"\n[Batch Test {i}] Lớp {grade}")
    print(f"Văn bản: {text[:50]}...")
    print(f"Tóm tắt: {summary}")

print("\n" + "=" * 50)
print("HOÀN TẤT BATCH TEST!")
print("=" * 50)

In [ ]:
# import pandas as pd

# df = pd.read_excel(r"/kaggle/working/datasets-dgtx/DATA_DG.xlsx")   # hoặc path đúng của bạn

# assert {"content", "summary", "grade"}.issubset(df.columns)
# df = df.dropna().reset_index(drop=True)


# from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# MODEL_NAME = "VietAI/vit5-base"

# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# print(tokenizer.model_max_length)  # ~512 tokens

# from datasets import Dataset

# MAX_INPUT_LEN = 256
# MAX_TARGET_LEN = 64

# def preprocess(batch):
#     prompts = [
#         f"Tóm tắt văn bản cho học sinh lớp {g}: {c}"
#         for c, g in zip(batch["content"], batch["grade"])
#     ]

#     model_inputs = tokenizer(
#         prompts,
#         truncation=True,
#         padding="max_length",
#         max_length=MAX_INPUT_LEN,
#     )

#     labels = tokenizer(
#         batch["summary"],
#         truncation=True,
#         max_length=MAX_TARGET_LEN,
#     )

#     model_inputs["labels"] = labels["input_ids"]
#     return model_inputs


# dataset = Dataset.from_pandas(df)
# dataset = dataset.train_test_split(test_size=0.1, seed=42)

# tokenized_ds = dataset.map(
#     preprocess,
#     batched=True,   # ⭐ BẮT BUỘC
#     remove_columns=dataset["train"].column_names
# )

# print(tokenized_ds["train"].column_names)


# import evaluate
# rouge = evaluate.load("rouge")
# bleu = evaluate.load("bleu")
# bertscore = evaluate.load("bertscore")

# def compute_metrics(eval_pred):
#     preds, labels = eval_pred

#     import numpy as np
    
#     if isinstance(preds, tuple):
#         preds = preds[0]
    
#     if isinstance(preds, np.ndarray):
#         if preds.ndim > 2:
#             preds = preds.reshape(preds.shape[0], -1)
#         preds = preds.tolist()
    
#     if isinstance(labels, np.ndarray):
#         labels = labels.tolist()

#     # Lấy vocab size để validate token IDs
#     vocab_size = len(tokenizer)
    
#     # Xử lý preds - flatten nếu cần và convert về integers hợp lệ
#     processed_preds = []
#     for pred in preds:
#         if isinstance(pred, np.ndarray):
#             pred = pred.tolist()
#         if isinstance(pred, list):
#             # Flatten nếu có nested lists
#             flat_pred = []
#             for item in pred:
#                 if isinstance(item, (list, np.ndarray)):
#                     flat_pred.extend(item)
#                 elif isinstance(item, (int, np.integer, float, np.floating)):
#                     token_id = int(item)
#                     # Chỉ giữ token IDs hợp lệ (0 <= token_id < vocab_size)
#                     if 0 <= token_id < vocab_size:
#                         flat_pred.append(token_id)
#             processed_preds.append(flat_pred if flat_pred else [tokenizer.pad_token_id])
#         else:
#             token_id = int(pred)
#             if 0 <= token_id < vocab_size:
#                 processed_preds.append([token_id])
#             else:
#                 processed_preds.append([tokenizer.pad_token_id])

#     # Xử lý label padding - chỉ giữ token IDs hợp lệ
#     processed_labels = []
#     for label in labels:
#         if isinstance(label, np.ndarray):
#             label = label.tolist()
#         valid_labels = [
#             int(l) for l in label 
#             if l != -100 
#             and isinstance(l, (int, np.integer, float, np.floating))
#             and 0 <= int(l) < vocab_size
#         ]
#         processed_labels.append(valid_labels if valid_labels else [tokenizer.pad_token_id])
    
#     decoded_preds = tokenizer.batch_decode(processed_preds, skip_special_tokens=True)
#     decoded_labels = tokenizer.batch_decode(processed_labels, skip_special_tokens=True)

#     # -------- ROUGE --------
#     rouge_result = rouge.compute(
#         predictions=decoded_preds,
#         references=decoded_labels
#     )

#     # -------- BLEU --------
#     bleu_result = bleu.compute(
#         predictions=[p.split() for p in decoded_preds],
#         references=[[l.split()] for l in decoded_labels]
#     )

#     # -------- BERTScore --------
#     bert_result = bertscore.compute(
#         predictions=decoded_preds,
#         references=decoded_labels,
#         lang="vi",
#         model_type="bert-base-multilingual-cased"
#     )

#     return {
#         "rouge1": rouge_result["rouge1"],
#         "rouge2": rouge_result["rouge2"],
#         "rougeL": rouge_result["rougeL"],

#         "bleu": bleu_result["bleu"],

#         "bertscore_precision": sum(bert_result["precision"]) / len(bert_result["precision"]),
#         "bertscore_recall": sum(bert_result["recall"]) / len(bert_result["recall"]),
#         "bertscore_f1": sum(bert_result["f1"]) / len(bert_result["f1"]),
#     }

# import optuna
# from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
# from transformers import EarlyStoppingCallback

# def objective(trial):

#     lr = trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True)
#     batch_size = 1
#     epochs = trial.suggest_int("num_train_epochs", 3, 5)
#     weight_decay = trial.suggest_float("weight_decay", 0.0, 0.1)

#     import torch
#     torch.cuda.empty_cache()

#     args = TrainingArguments(
#         output_dir=f"./optuna_trial_{trial.number}",

#         eval_strategy="steps",
#         eval_steps=500,
#         logging_strategy="steps",
#         logging_steps=100,
#         save_strategy="no",

#         learning_rate=lr,
#         per_device_train_batch_size=batch_size,
#         per_device_eval_batch_size=1,
#         gradient_accumulation_steps=16,
#         num_train_epochs=epochs,
#         weight_decay=weight_decay,

#         # 🔥 CHỐNG OVERFITTING
#         label_smoothing_factor=0.1,
#         max_grad_norm=1.0,
#         fp16=True,
#         gradient_checkpointing=True,
#         dataloader_num_workers=0,
#         dataloader_pin_memory=False,
#         ddp_find_unused_parameters=False,

#         metric_for_best_model="eval_rougeL",
#         greater_is_better=True,

#         report_to="none"
#     )

#     model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
#     data_collator = DataCollatorForSeq2Seq(
#         tokenizer=tokenizer,
#         model=model,
#         padding=True,
#         pad_to_multiple_of=None
#     )

#     trainer = Trainer(
#         model=model,
#         args=args,
#         train_dataset=tokenized_ds["train"],
#         eval_dataset=tokenized_ds["test"],
#         tokenizer=tokenizer,
#         data_collator=data_collator,
#         compute_metrics=compute_metrics,
#         callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
#     )

#     trainer.train()
#     metrics = trainer.evaluate()
    
#     result = metrics["eval_rougeL"]
    
#     del model
#     del trainer
#     del data_collator
#     torch.cuda.empty_cache()
    
#     return result

# import pandas as pd

# def save_metrics(trainer, filename="metrics.csv"):
#     logs = trainer.state.log_history
#     df_metrics = pd.DataFrame(logs)
#     df_metrics.to_csv(filename, index=False)
#     return df_metrics



# study = optuna.create_study(direction="maximize")
# study.optimize(objective, n_trials=3)

# best_params = study.best_params
# print(best_params)

# final_args = TrainingArguments(
#     output_dir="./vit5_final",

#     eval_strategy="epoch",
#     save_strategy="epoch",
#     logging_strategy="epoch",

#     learning_rate=best_params["learning_rate"],
#     per_device_train_batch_size=1,
#     per_device_eval_batch_size=1,
#     gradient_accumulation_steps=16,
#     num_train_epochs=best_params["num_train_epochs"],
#     weight_decay=best_params["weight_decay"],

#     # 🔒 ANTI-OVERFITTING
#     label_smoothing_factor=0.1,
#     max_grad_norm=1.0,
#     fp16=True,
#     gradient_checkpointing=True,
#     dataloader_num_workers=0,
#     dataloader_pin_memory=False,
#     ddp_find_unused_parameters=False,

#     load_best_model_at_end=True,
#     metric_for_best_model="eval_rougeL",
#     greater_is_better=True,

#     save_total_limit=2,
#     report_to="none"
# )

# import torch
# torch.cuda.empty_cache()

# final_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
# final_data_collator = DataCollatorForSeq2Seq(
#     tokenizer=tokenizer,
#     model=final_model,
#     padding=True,
#     pad_to_multiple_of=None
# )

# final_trainer = Trainer(
#     model=final_model,
#     args=final_args,
#     train_dataset=tokenized_ds["train"],
#     eval_dataset=tokenized_ds["test"],
#     tokenizer=tokenizer,
#     data_collator=final_data_collator,
#     compute_metrics=compute_metrics,
#     callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
# )

# final_trainer.train()

# final_trainer.save_model("./vit5_grade_summary")
# tokenizer.save_pretrained("./vit5_grade_summary")

# metrics_df = save_metrics(final_trainer, "vit5_metrics.csv")
# metrics_df.head()

# import matplotlib.pyplot as plt

# plt.figure(figsize=(10,6))

# plt.plot(metrics_df["epoch"], metrics_df["eval_rouge1"], label="ROUGE-1")
# plt.plot(metrics_df["epoch"], metrics_df["eval_rougeL"], label="ROUGE-L")
# plt.plot(metrics_df["epoch"], metrics_df["bertscore_f1"], label="BERTScore-F1")

# plt.xlabel("Epoch")
# plt.ylabel("Score")
# plt.title("Evaluation Metrics over Epochs")
# plt.legend()
# plt.grid(True)
# plt.show()